# Final Dataset Assembly
Pipeline position: Stage 3 of 3

This notebook combines:
1. Naive search results with keywords (from Stage 2: 02_searchterm_identification.ipynb)
2. Manual search results (Zotero exports)

**Output:** Final deduplicated dataset ready for analysis.



## 1. Imports and load intermediate results

In [ ]:
from pathlib import Path
from datetime import date
import pandas as pd

# Define paths (relative to notebooks/ folder)
intermediate_dir = Path("../02_processed/intermediate")
manual_dir = Path("../01_raw/manual")
output_dir = Path("../02_processed/combined")

# Standard columns (matching scopus schema from Stage 2)
standard_cols = [
    "title", "abstract", "eid", "authors", "journal", "issue",
    "volume", "keywords", "doctype", "year", "doi",
    "pagestart", "pageend"
]

In [ ]:
# Load intermediate results from Stage 2
# Find most recent file (sorted by filename which includes date)
naive_files = sorted(intermediate_dir.glob("*_naive_search_with_keywords.csv"))

if naive_files:
    latest_file = naive_files[-1]  # most recent
    naive_results_df = pd.read_csv(latest_file)
    print(f"Loaded naive search results from: {latest_file.name}")
    print(f"Entries: {len(naive_results_df)}")
else:
    raise FileNotFoundError(f"No naive search results found in {intermediate_dir}. Run Stage 2 first.")

## 2. Import manual search results (Zotero exports)

In [ ]:
# Column mapping: Zotero export → standard schema
zotero_to_standard = {
    "Title": "title",
    "Abstract Note": "abstract",
    "Key": "eid",              # Zotero key as identifier
    "Author": "authors",
    "Publication Title": "journal",
    "Issue": "issue",
    "Volume": "volume",
    "Item Type": "doctype",
    "Publication Year": "year",
    "DOI": "doi",
}

manual_dfs = []

for csv_path in manual_dir.glob("manualsearch_*.csv"):
    df = pd.read_csv(csv_path)
    
    # Rename columns to standard schema
    df = df.rename(columns=zotero_to_standard)
    
    # Split Pages into pagestart/pageend
    if "Pages" in df.columns:
        pages_split = df['Pages'].str.split('-', expand=True)
        df['pagestart'] = pages_split[0] if 0 in pages_split.columns else None
        df['pageend'] = pages_split[1] if 1 in pages_split.columns else None
    
    # Add metadata columns
    df["database"] = "manual"
    search_run = "manual_" + csv_path.stem.replace("manualsearch_", "")
    df["search_run"] = search_run
    
    # Normalize title and abstract
    df["title_normalized"] = df["title"].str.lower().str.strip()
    df["abs_normalized"] = df["abstract"].str.lower().str.strip() if "abstract" in df.columns else None
    
    # Keep only columns that exist
    keep_cols = [c for c in standard_cols + ["database", "search_run", "title_normalized", "abs_normalized"] 
                 if c in df.columns]
    df = df[keep_cols]
    
    manual_dfs.append(df)
    print(f"  Loaded {csv_path.name}: {len(df)} entries")

if manual_dfs:
    manual_results_df = pd.concat(manual_dfs, ignore_index=True)
    print(f"\nTotal manual search entries: {len(manual_results_df)} from {len(manual_dfs)} files")
else:
    manual_results_df = pd.DataFrame()
    print("No manual search files found.")

## 3. Merge and deduplicate

In [7]:
# Combine dataframes
combined_df = pd.concat([naive_results_df, manual_results_df], ignore_index=True)
print(f"Combined: {len(combined_df)} entries before deduplication")

# First deduplication: by DOI
# Prioritize scopus/wos entries over manual (ascending=True puts False/_is_manual=False first)
combined_df['_is_manual'] = combined_df['database'] == 'manual'

# Deduplicate rows with DOI
dedup_with_doi = (
    combined_df[combined_df['doi'].notna()]
    .sort_values('_is_manual', ascending=True)
    .drop_duplicates(subset='doi', keep='first')
)

# Keep rows without DOI
missing_doi = combined_df[combined_df['doi'].isna()]

# Combine
combined_df = pd.concat([dedup_with_doi, missing_doi], ignore_index=True)
print(f"After DOI deduplication: {len(combined_df)} entries")

# Second deduplication: by title
# Prioritize: 1) entries with DOI, 2) scopus/wos over manual
combined_df['_has_doi'] = combined_df['doi'].notna()

final_results_df = (
    combined_df
    .sort_values(['_has_doi', '_is_manual'], ascending=[False, True])
    .drop_duplicates(subset='title_normalized', keep='first')
    .drop(columns=['_has_doi', '_is_manual'])
    .reset_index(drop=True)
)

print(f"After title deduplication: {len(final_results_df)} entries")

# Summary statistics
print(f"\nDataset composition:")
print(final_results_df['database'].value_counts())

Combined: 512 entries before deduplication
After DOI deduplication: 503 entries
After title deduplication: 502 entries

Dataset composition:
database
scopus    252
wos       203
manual     47
Name: count, dtype: int64


## 4. Save final outputs

In [8]:
# Create output directory if needed
output_dir.mkdir(parents=True, exist_ok=True)

# Date prefix for file naming
today = date.today().strftime("%Y-%m-%d")

# Save main results
main_output = output_dir / f"{today}_final_results.csv"
final_results_df.to_csv(main_output, index=False)
print(f"Saved: {main_output.name} ({len(final_results_df)} entries)")

# Load and save welfare state change subset from Stage 2
wschange_files = sorted(intermediate_dir.glob("*_wschange_naive.csv"))
if wschange_files:
    finres_wschange = pd.read_csv(wschange_files[-1])
    wschange_output = output_dir / f"{today}_wschange_subset.csv"
    finres_wschange.to_csv(wschange_output, index=False)
    print(f"Saved: {wschange_output.name} ({len(finres_wschange)} entries)")
else:
    print("No welfare state change subset found in intermediate folder.")

print(f"\n✓ Final dataset ready for analysis in ../03_analysis/")

Saved: 2026-01-27_final_results.csv (502 entries)
Saved: 2026-01-27_wschange_subset.csv (151 entries)

✓ Final dataset ready for analysis in ../03_analysis/


## 5. Quick data inspection

In [9]:
# Preview the final dataset
print(f"Final dataset shape: {final_results_df.shape}")
print(f"\nColumns: {list(final_results_df.columns)}")
print(f"\nYear range: {final_results_df['year'].min()} - {final_results_df['year'].max()}")
print(f"\nSearch runs included:")
print(final_results_df['search_run'].value_counts())

Final dataset shape: (502, 21)

Columns: ['title', 'abstract', 'eid', 'authors', 'journal', 'issue', 'volume', 'keywords', 'doctype', 'year', 'doi', 'pagestart', 'pageend', 'database', 'search_run', 'title_normalized', 'abs_normalized', 'is_globalsouth', 'title_abstract', 'keybert', 'yake']

Year range: 1959 - 2026

Search runs included:
search_run
socialpolicy_covid                                                                 82
globalsouth_countrylist_inequality_poverty_covid_greatrecession                    75
welfarestate_crisis                                                                72
resilience                                                                         67
socialpolicy_crisis                                                                62
welfarestate_covid                                                                 41
manual_wschange                                                                    14
manual_resilience                              

In [10]:
# Show sample of manual entries
print("Sample of manual search entries:")
final_results_df[final_results_df['database'] == 'manual'][['title', 'year', 'search_run']].head(10)

Sample of manual search entries:


,title,year,search_run
408,Chancen des Resilienzbegriffs für eine soziolo...,2015,manual_resilience
409,Families with low resources striving for resil...,2025,manual_resilience
410,Fostering resilience: Welfare policy change ac...,2024,manual_resilience
411,Families with low resources striving for resil...,2025,manual_resilience
412,Revising the concept of resilience for applica...,2025,manual_resilience
413,Families with low resources striving for resil...,2025,manual_resilience
414,"Conclusion to Families, Welfare States and Res...",2025,manual_resilience
415,Disruptiver sozialer Wandel und das Problem de...,2015,manual_resilience
416,Families with low resources striving for resil...,2025,manual_resilience
417,Families with low resources striving for resil...,2025,manual_resilience
